# **LLM (google-gemma) Turkish news headlines text classifier**

In [ ]:
print("Installing compatible PyTorch version (for CUDA 12.1)...")
!pip install "torch==2.3.0" --index-url https://download.pytorch.org/whl/cu121

print("Installing other dependencies...")
!pip install -U "transformers==4.40.0" "datasets==2.18.0" "peft==0.10.0" "trl==0.8.6" "bitsandbytes==0.43.0" "accelerate==0.29.3"

# Install the correct triton version
!pip install triton==2.2.0

# Install pandas for CSV loading
!pip install pandas

print("✅ All libraries installed.")
print("‼️  IMPORTANT: Now go to 'Runtime > Restart session...' and run the *next* cell.")

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model
from datasets import Dataset
from trl import SFTTrainer
import os

# to download gemma, need to create a READ-ACCESS TOKEN on huggingface
from huggingface_hub import notebook_login
notebook_login()


In [3]:
import pandas as pd

df = pd.read_csv("/content/TurkishHeadlines.csv")

df.columns = ['text', 'label']
df = df.dropna(subset=['text', 'label']) # Drop any rows with missing data
print(f"Total rows: {len(df)}")
print("Categories found:")
print(df['label'].value_counts())

def format_prompt(example):
    text = example['text']
    label = example['label']
    # This is the simple key-value format
    return {
        "formatted_text": f"<start_of_turn>user\nMetin: \"{text}\"\nKategori:<end_of_turn>\n<start_of_turn>model\n{label}<end_of_turn>"
    }

print("Formatting and splitting data...")
dataset = Dataset.from_pandas(df)

# Split 90% for training, 10% for evaluation
train_test_split = dataset.train_test_split(test_size=0.1)
train_dataset = train_test_split['train']
eval_dataset = train_test_split['test']

train_dataset = train_dataset.map(format_prompt, remove_columns=['text', 'label'])
eval_dataset = eval_dataset.map(format_prompt, remove_columns=['text', 'label'])

print("✅ Data is ready!")
print(f"Training examples: {len(train_dataset)}")
print(f"Evaluation examples: {len(eval_dataset)}")
print("--- Sample Prompt ---")
print(train_dataset[0]['formatted_text'])

Total rows: 4200
Categories found:
label
Ekonomi      600
Magazin      600
Sağlık       600
Siyaset      600
Spor         600
Teknoloji    600
Yaşam        600
Name: count, dtype: int64
Formatting and splitting data...


Map:   0%|          | 0/3780 [00:00<?, ? examples/s]

Map:   0%|          | 0/420 [00:00<?, ? examples/s]

✅ Data is ready!
Training examples: 3780
Evaluation examples: 420
--- Sample Prompt ---
<start_of_turn>user
Metin: "Bütçe şubatta 1.4 milyar açık verdi"
Kategori:<end_of_turn>
<start_of_turn>model
Ekonomi<end_of_turn>


In [4]:
model_id = "google/gemma-2b-it" # The instruction-tuned model

# --- Quantization Config ---
# This configures the model to load in 4-bit precision
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4", # "nf4" (Normalized Float 4)
    bnb_4bit_use_double_quant=True, # a second quantization for even more memory savings
    bnb_4bit_compute_dtype=torch.bfloat16 # bfloat16 for computations
)

# --- Load Model ---
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto" # Automatically maps layers to GPU/CPU
)

# --- Load Tokenizer ---
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token # Set pad token to end-of-sentence token
tokenizer.padding_side = "right"

print("✅ Model and Tokenizer loaded in 4-bit!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/627 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/13.5k [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.95G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/67.1M [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/34.2k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

✅ Model and Tokenizer loaded in 4-bit!


In [5]:
# --- Prepare model for k-bit training ---
# This freezes the original 4-bit model and prepares it for LoRA
model = prepare_model_for_kbit_training(model)

# --- LoRA Config ---
peft_config = LoraConfig(
    r=16,  # The "rank" of the LoRA matrices. Higher is more params, but 16 is a good default.
    lora_alpha=32, # A scaling factor. (alpha / r) is the scaling.
    lora_dropout=0.05, # Dropout for regularization
    bias="none",
    task_type="CAUSAL_LM",
    # Target the attention layers of the Gemma model
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)

# Attach LoRA adapters to the model
model = get_peft_model(model, peft_config)

print("✅ LoRA adapters attached to the model.")
model.print_trainable_parameters() # See how few params we're training!

✅ LoRA adapters attached to the model.
trainable params: 19,611,648 || all params: 2,525,784,064 || trainable%: 0.7764578247018349


In [ ]:
print("Setting up Trainer...")
training_args = TrainingArguments(
    output_dir="turkish-headline-classifier",
    report_to="none",
    num_train_epochs=2,  # <-- With 4K data, 2 epochs is perfect.
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4, # Simulate a larger batch size
    optim="paged_adamw_8bit",
    logging_steps=50,
    learning_rate=2e-5,
    fp16=False,
    bf16=True,
    evaluation_strategy="steps", # <-- We can now evaluate!
    eval_steps=100,              # <-- Check performance every 100 steps
    save_steps=100,              # <-- Save checkpoint every 100 steps
    max_grad_norm=0.3,
    warmup_ratio=0.03,
    lr_scheduler_type="constant",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset, # 90%
    eval_dataset=eval_dataset,   # 10%
    peft_config=peft_config,
    dataset_text_field="formatted_text",
    max_seq_length=512,
    args=training_args,
)

print("🚀 Starting real training...")
trainer.train()

print("🏁 Training complete!")

# --- Save the final adapter ---
adapter_model_name = "gemma-turkish-classifier-final"
trainer.model.save_pretrained(adapter_model_name)
tokenizer.save_pretrained(adapter_model_name)

print(f"✅ Final LoRA adapter saved to '{adapter_model_name}'")

Setting up Trainer...


Map:   0%|          | 0/3780 [00:00<?, ? examples/s]

Map:   0%|          | 0/420 [00:00<?, ? examples/s]

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


🚀 Starting real training...


/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:464: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.4 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  warnings.warn(


Step,Training Loss,Validation Loss
100,3.334600,2.919348
200,2.553900,2.500182


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:464: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.4 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download

In [ ]:
from peft import PeftModel

print("Loading base model...")
base_model = AutoModelForCausalLM.from_pretrained(
    model_id, # "google/gemma-2b-it"
    quantization_config=bnb_config,
    device_map="auto"
)

print("Loading adapter...")
adapter_path = "gemma-turkish-classifier-final" # <-- Your saved adapter
model = PeftModel.from_pretrained(base_model, adapter_path)

# The correct label for this headline is "Ekonomi"
new_text = "Tüpraş 7 yıl vadeyle 700 milyon dolar borçlanıyor"

# Format it just like the training data
prompt = f"<start_of_turn>user\nMetin: \"{new_text}\"\nKategori:<end_of_turn>\n<start_of_turn>model\n"
inputs = tokenizer(prompt, return_tensors="pt", return_attention_mask=True).to("cuda")

print("Generating prediction...")
outputs = model.generate(
    **inputs,
    max_new_tokens=10, # 10 tokens is enough for any label
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.eos_token_id
)

prediction = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

print("\n" + "="*30)
print(f"Test Metni: {new_text}")
print(f"Modelin Tahmini: {prediction.strip()}")
print("="*30)